In [24]:
import sys
import gc
import os
import time

import numpy as np
import pandas as pd

from aeon.datasets import load_classification
from aeon.classification.convolution_based import RocketClassifier
from aeon.classification.interval_based import QUANTClassifier
from aeon.classification.distance_based import KNeighborsTimeSeriesClassifier

from sklearn.metrics import accuracy_score

In [25]:
RANDOM_STATE = 42

In [26]:
# List all high dimensional datasets (with more than 1000 dimensions) in the UCR archive.
high_dim_datasets = [
  'ACSF1',
  # 'CinCECGTorso',
  # 'EOGHorizontalSignal',
  # 'EOGVerticalSignal',
  # 'EthanolLevel',
  # 'HandOutlines',
  # 'Haptics',
  # 'HouseTwenty',
  # 'InlineSkate',
  # 'Mallat',
  # 'MixedShapesRegularTrain',
  # 'MixedShapesSmallTrain',
  # 'Phoneme',
  # 'PigAirwayPressure',
  # 'PigArtPressure',
  # 'PigCVP',
  # 'Rock',
  # 'SemgHandGenderCh2',
  # 'SemgHandMovementCh2',
  # 'SemgHandSubjectCh2',
  # 'StarLightCurves',
]

In [27]:
classifiers = {
  "Rocket": RocketClassifier(random_state=RANDOM_STATE),
  # "QUANT": QUANTClassifier(random_state=RANDOM_STATE),
  # "1NN-DTW": KNeighborsTimeSeriesClassifier(n_neighbors=1, distance="dtw"),
  # 'LITE'
}

In [28]:
retention_rates = [
  0.85,
  # 0.70, 
  # 0.55,
  # 0.40,
  # 0.25,
  ]

In [29]:
# ---------- LOCATION ----------
def agg_average(x):
    return np.mean(x)

def agg_median(x):
    return np.median(x)

def agg_first(x):
    return x[0]

def agg_last(x):
    return x[-1]

def agg_central(x):
    return x[len(x)//2]


# ---------- DISPERSION ----------
def agg_variance(x):
    return np.var(x)

def agg_std(x):
    return np.std(x)

def agg_iqr(x):
    return np.percentile(x, 75) - np.percentile(x, 25)

def agg_range(x):
    return np.max(x) - np.min(x)


# ---------- EXTREMA ----------
def agg_max(x):
    return np.max(x)

def agg_min(x):
    return np.min(x)

def agg_avg_max(x):
    return np.mean(x) - np.max(x)

def agg_avg_min(x):
    return np.mean(x) - np.min(x)


# ---------- ENERGY ----------
def agg_energy(x):
    return np.sum(x**2)

def agg_rms(x):
    return np.sqrt(np.mean(x**2))


# ---------- TREND ----------
def agg_slope(x):
    t = np.arange(len(x))
    if len(x) < 2:
        return 0
    # simple linear regression
    cov = np.cov(t, x, bias=True)[0, 1]
    var_t = np.var(t)
    return cov / var_t if var_t > 0 else 0

def agg_endpoint_diff(x):
    return x[-1] - x[0]

AGG_FUNCS = {
    "Average": agg_average,
    # "Median": agg_median,
    # "First": agg_first,
    # "Central": agg_central,
    # "Last": agg_last,
    # "Variance": agg_variance,
    # "Std": agg_std,
    # "IQR": agg_iqr,
    # "Max-Min": agg_range,
    # "Max": agg_max,
    # "Min": agg_min,
    # "Avg-Max": agg_avg_max,
    # "Avg-Min": agg_avg_min,
    # "Energy": agg_energy,
    # "RMS": agg_rms,
    # "Slope": agg_slope,
    # "Last-First diff": agg_endpoint_diff,
}

def PAA_reduce(s, w, agg="Average"):
    n = len(s)
    s = np.array(s)

    idx = np.floor(np.linspace(0, w, n, endpoint=False)).astype(int)

    f = AGG_FUNCS[agg]
    res = [f(s[idx == i]) for i in range(w)]

    # res = znorm(res)
    return np.array(res)

In [30]:
def znorm(x):
  x_znorm = (x - np.mean(x)) / np.std(x)
  return x_znorm

In [31]:
def load_and_normalize_dataset(dataset_name):
    # Carrega o dataset
    print(f'Carregando dataset {dataset_name}...')
    X_train, y_train = load_classification(dataset_name, split='train')
    X_test, y_test = load_classification(dataset_name, split='test')
    print(f'Dataset {dataset_name} carregado com sucesso. Tamanho do treino: {X_train.shape}, Tamanho do teste: {X_test.shape}')

    # Normaliza os dados usando Z-normalization
    print('Normalizando os dados de treino e teste (Z-normalization)...')
    X_train_normalized = np.array([[znorm(series) for series in sample] for sample in X_train])
    X_test_normalized = np.array([[znorm(series) for series in sample] for sample in X_test])
    print('Normalização concluída!')
    return X_train_normalized, y_train, X_test_normalized, y_test

In [32]:
def train_and_evaluate_classifier(clf_name, clf, X_train, y_train, X_test, y_test):
    # Treina o classificador
    print(f'Treinando o classificador {clf_name}...')
    start = time.time()
    clf.fit(X_train, y_train)
    end = time.time()
    duration = np.round(end - start, 2)
    print(f'Classificador treinado com sucesso em {duration} segundos!')

    # Avalia o classificador
    print(f'Avaliando o classificador {clf_name}...')
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy:.4f}')

    return accuracy, duration

In [33]:
def reduce_dimensionality(method_name, method, X_train, X_test, reduction_rate):
    series_size = X_train.shape[2]
    w = np.round(series_size * (reduction_rate)).astype(int)
    print(f'\nIniciando redução com {method_name} (Novo tamanho: {w})')

    start_time = time.time()

    def run_with_progress(data, label):
        reduced_data = []
        total = len(data)
        for i, sample in enumerate(data):
            reduced_sample = []
            for j, series in enumerate(sample):
                res = method(series, w)
                reduced_sample.append(res)

            reduced_data.append(reduced_sample)
            
            # --- BARRA DE PROGRESSO MANUAL ---
            percent = (i + 1) / total
            bar_length = 30
            done = int(percent * bar_length)
            bar = "█" * done + "-" * (bar_length - done)
            sys.stdout.write(f'\r{label}: [{bar}] {percent:>.1%} ({i+1}/{total})')
            sys.stdout.flush()
            # ----------------------------------
            
        print() # Quebra de linha ao finalizar
        return np.array(reduced_data)

    X_train_reduced = run_with_progress(X_train, "Treino")
    X_test_reduced = run_with_progress(X_test, "Teste ")
    
    end_time = time.time()
    duration = np.round(end_time - start_time, 2)
    print(f'Redução concluída em {duration} segundos!')

    return X_train_reduced, X_test_reduced, duration

In [34]:
########################################################################################################################
#                                                     RODAGEM DE TESTES                                                #
########################################################################################################################
results = []
output_file = 'results_classification.csv'

for dataset in high_dim_datasets:
    try:
        X_train, y_train, X_test, y_test = load_and_normalize_dataset(dataset)
        
        # 1. Dados Originais
        for clf_name, clf in classifiers.items():
            acc, duration = train_and_evaluate_classifier(clf_name, clf, X_train, y_train, X_test, y_test)
            res = {
                'dataset': dataset, 
                'classifier': clf_name, 
                'operator': None,
                'retention_rate': None, 
                'series_size': X_train.shape[2], 
                'accuracy': acc, 
                'training_time': duration,
                'reduction_time': 0
            }
            results.append(res)
            pd.DataFrame([res]).to_csv(output_file, mode='a', header=not os.path.exists(output_file), index=False)

        # 2. Dados Reduzidos
        for agg_name, agg_func in AGG_FUNCS.items():
            for rate in retention_rates:
                method_to_call = lambda s, w: PAA_reduce(s, w, agg=agg_name)
                X_train_red, X_test_red, reduction_duration = reduce_dimensionality(agg_name, method_to_call, X_train, X_test, rate)
                
                for clf_name, clf in classifiers.items():
                    acc, duration = train_and_evaluate_classifier(clf_name, clf, X_train_red, y_train, X_test_red, y_test)
                    res = {
                        'dataset': dataset, 
                        'classifier': clf_name, 
                        'operator': agg_name,
                        'retention_rate': rate, 
                        'series_size': X_train_red.shape[2], 
                        'accuracy': acc, 
                        'training_time': duration,
                        'reduction_time': reduction_duration
                    }
                    results.append(res)
                    # Salva cada linha imediatamente para segurança
                    pd.DataFrame([res]).to_csv(output_file, mode='a', header=False, index=False)

                # Limpeza pesada após cada taxa de redução/método
                del X_train_red, X_test_red
                gc.collect()

        # Limpeza após processar um dataset inteiro
        del X_train, X_test, y_train, y_test
        gc.collect()
        
    except Exception as e:
        print(f"Erro ao processar dataset {dataset}: {e}")
        continue # Pula para o próximo se um falhar

Carregando dataset ACSF1...
Dataset ACSF1 carregado com sucesso. Tamanho do treino: (100, 1, 1460), Tamanho do teste: (100, 1, 1460)
Normalizando os dados de treino e teste (Z-normalization)...
Normalização concluída!
Treinando o classificador Rocket...
Classificador treinado com sucesso em 10.38 segundos!
Avaliando o classificador Rocket...
Accuracy: 0.8900

Iniciando redução com Average (Novo tamanho: 1241)
Treino: [██████████████████████████████] 100.0% (100/100)
Teste : [██████████████████████████████] 100.0% (100/100)
Redução concluída em 2.55 segundos!
Treinando o classificador Rocket...
Classificador treinado com sucesso em 8.79 segundos!
Avaliando o classificador Rocket...
Accuracy: 0.8700
